# Please use this for torrent download


In [ ]:
#Grey Knight Tech ©️ Combined Code for Multi-Torrent Download with Progress Monitoring

# --- Step 1: Install necessary libraries and aria2c ---
!apt-get -qq install -y aria2
!pip install aria2p tqdm

print("✅ Installed required libraries and aria2c.")

# --- Step 2: Mount Google Drive ---
from google.colab import drive
drive.mount('/content/drive')

# --- Step 3: Set download directory in Google Drive ---
import os

# Change this path if you want another folder in Drive
download_dir = "/content/drive/My Drive/torrent_download"
os.makedirs(download_dir, exist_ok=True)

print("✅ Download directory set to:", download_dir)

# --- Step 4: Start aria2c with RPC enabled ---
import subprocess
import time

# Start aria2c daemon with RPC enabled
aria2_rpc_port = 6800  # You can change this port if needed
aria2_command = [
    "aria2c",
    "--enable-rpc",
    f"--rpc-listen-port={aria2_rpc_port}",
    "--rpc-listen-all",
    "--seed-time=0",
    "--console-log-level=warn",
    "-d", download_dir,
    "--continue=true", # Continue downloading partially downloaded files
    "--allow-overwrite=true", # Allow overwriting existing files
]

# Use subprocess.Popen to run aria2c in the background
# Redirect stdout and stderr to avoid cluttering the output directly
aria2_process = subprocess.Popen(
    aria2_command,
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
    text=True
)

# Give aria2c a moment to start
time.sleep(2)

# Check if aria2c process started successfully
if aria2_process.poll() is None:
    print(f"✅ aria2c daemon started successfully on port {aria2_rpc_port}")
else:
    print("❌ Failed to start aria2c daemon.")
    # Print any error output
    stderr_output = aria2_process.stderr.read()
    if stderr_output:
        print("aria2c stderr:\n", stderr_output)
    # Exit if aria2c failed to start
    exit()


# --- Step 5: Collect multiple magnet links ---
magnet_links = []
print("\nPlease enter the magnet links you want to download, one by one.")
print("Enter 'done' when you have finished adding links.")

while True:
    link = input(f"🔗 Enter magnet link {len(magnet_links) + 1} (or 'done'): ").strip()
    if link.lower() == 'done':
        break
    if link and link.startswith("magnet:?"):
        magnet_links.append(link)
        print("✅ Magnet link added.")
    elif link:
        print("❌ Invalid magnet link. Please make sure it starts with 'magnet:?'.")

if not magnet_links:
    print("\nNo magnet links provided. Exiting.")
    # Terminate aria2c process if no links were added
    if aria2_process.poll() is None:
        aria2_process.terminate()
    exit()

print(f"\nCollected {len(magnet_links)} magnet link(s).")


# --- Step 6: Add torrents via RPC ---
import aria2p

# Initialize aria2p client
# The default host is localhost and the default port is 6800
aria2 = aria2p.API(
    aria2p.Client(
        host="http://localhost",
        port=aria2_rpc_port,
        secret="" # No secret token is set in the aria2_command
    )
)

added_count = 0
error_count = 0

print("\nAdding torrents to aria2c...")

for link in magnet_links:
    try:
        result = aria2.add_magnet(link)
        if result:
            print(f"✅ Successfully added torrent: {result.name or result.gid}")
            added_count += 1
        else:
            print(f"❌ Failed to add torrent: {link}")
            error_count += 1
    except Exception as e:
        print(f"❌ Error adding torrent {link}: {e}")
        error_count += 1

print(f"\nSummary: {added_count} torrent(s) added successfully, {error_count} error(s).")

if added_count == 0:
    print("No torrents were added to monitor. Exiting.")
    # Terminate aria2c process if no links were successfully added
    if aria2_process.poll() is None:
        aria2_process.terminate()
    exit()

print("🚀 Starting download monitoring...")

# --- Step 7: Monitor download progress and display GUI-like progress ---
from tqdm.notebook import tqdm
import datetime

# Define progress_bars and completed_gids outside the function to maintain state
progress_bars = {}
completed_gids = set()

def get_downloads_status(api):
    """Fetches the status of all active, waiting, and paused downloads."""
    try:
        return api.get_downloads()
    except Exception as e:
        # Handle potential connection errors to aria2c
        print(f"Error fetching download status: {e}")
        return []


def display_progress(downloads):
    """Displays progress bars for each download."""
    if not downloads:
        # No downloads to display, but aria2c might still be running with completed/errored ones
        return

    global progress_bars, completed_gids # Access the global variables

    for download in downloads:
        gid = download.gid
        status = download.status
        total_length = download.total_length
        completed_length = download.completed_length
        download_speed = download.download_speed
        eta = download.eta

        # Convert bytes to MiB for display
        total_length_mib = total_length / (1024 * 1024) if total_length else 0
        completed_length_mib = completed_length / (1024 * 1024) if completed_length else 0
        download_speed_mibps = download_speed / (1024 * 1024) if download_speed else 0

        # Determine the state of the download
        is_complete = status == 'complete'
        is_error = status == 'error'
        is_removed = status == 'removed'

        if is_complete and gid not in completed_gids:
            # Mark as completed and close the progress bar
            if gid in progress_bars:
                progress_bars[gid].n = total_length # Set to total for final display
                progress_bars[gid].refresh()
                progress_bars[gid].close()
                del progress_bars[gid]
            print(f"\n✅ Download complete: {download.name or download.gid}")
            completed_gids.add(gid)
            continue # Move to the next download

        if is_error or is_removed:
             if gid in progress_bars:
                progress_bars[gid].n = completed_length # Show current progress before closing
                progress_bars[gid].refresh()
                progress_bars[gid].close()
                del progress_bars[gid]
             if is_error:
                 print(f"\n❌ Download error for {download.name or download.gid}. Status: {status}, Error Code: {download.error_code}, Error Message: {download.error_message}")
             elif is_removed:
                  print(f"\nℹ️ Download removed: {download.name or download.gid}")
             if gid in completed_gids:
                 completed_gids.remove(gid) # Remove from completed if it somehow ended up there
             continue


        if gid not in progress_bars:
            # Create a new progress bar for a new download
            bar_description = f"{download.name or download.gid}"
            if total_length > 0:
                 progress_bars[gid] = tqdm(total=total_length, unit='B', unit_scale=True, desc=bar_description, position=len(progress_bars), leave=True)
            else:
                 # Handle case where total_length is not yet known (e.g., metadata download)
                 progress_bars[gid] = tqdm(unit='B', unit_scale=True, desc=bar_description + " (metadata)", position=len(progress_bars), leave=True)


        # Update the existing progress bar
        bar = progress_bars[gid]
        bar.n = completed_length

        # Convert eta (timedelta) to seconds if it exists
        eta_seconds = eta.total_seconds() if isinstance(eta, datetime.timedelta) else None

        bar.set_postfix_str(f"Speed: {download_speed_mibps:.2f} MiB/s, ETA: {time.strftime('%H:%M:%S', time.gmtime(int(eta_seconds))) if eta_seconds is not None else 'N/A'}")
        bar.refresh()

# --- Monitoring loop ---
print("Monitoring downloads... Press the stop button to interrupt.")

try:
    while True:
        downloads = get_downloads_status(aria2)

        # Check and stop already completed downloads at the start of the loop
        for download in list(downloads): # Iterate over a copy as we might remove items
            if download.status == 'complete' and download.gid not in completed_gids:
                 print(f"\n✅ Download {download.name or download.gid} is already complete. Stopping seeding.")
                 try:
                     aria2.stop(download.gid) # Stop seeding
                     aria2.remove(download.gid) # Remove from list
                     completed_gids.add(download.gid) # Mark as completed
                 except Exception as stop_e:
                     print(f"Error stopping/removing completed download {download.gid}: {stop_e}")


        # Filter for downloads that are still in a state we want to monitor progress for
        active_monitor_downloads = [d for d in downloads if d.status in ['active', 'waiting', 'paused', 'downloading']]

        # Also include downloads that were just added but haven't started yet,
        # or downloads that might transition from complete/error back to active (though less common)
        # Ensure we don't try to monitor downloads we've marked as completed and removed
        downloads_to_display = [d for d in downloads if d.gid not in completed_gids]


        if not active_monitor_downloads and all(d.status in ['complete', 'error', 'removed'] for d in downloads_to_display):
             # Check if all downloads that were initially added are now in a final state
             # This handles cases where downloads might finish quickly before the first monitoring loop iteration
             print("\nAll downloads have finished or are in a final state.")
             break # Exit the loop if all downloads are finished

        display_progress(downloads_to_display) # Pass only downloads we are actively monitoring/displaying

        time.sleep(1)  # Update every second

except KeyboardInterrupt:
    print("\nMonitoring interrupted by user.")
except Exception as e:
    print(f"\nAn error occurred during monitoring: {e}")
finally:
    # Ensure all progress bars are closed when the loop exits
    if progress_bars:
        for gid, bar in list(progress_bars.items()): # Use list() to iterate over a copy
            bar.close()
            del progress_bars[gid] # Clean up the dictionary

    # Terminate the aria2c process when monitoring stops
    if aria2_process.poll() is None:
        print("\nTerminating aria2c process...")
        aria2_process.terminate()
        aria2_process.wait()
        print("aria2c process terminated.")

    print("Monitoring stopped.")

Selecting previously unselected package libc-ares2:amd64.
(Reading database ... 126435 files and directories currently installed.)
Preparing to unpack .../libc-ares2_1.18.1-1ubuntu0.22.04.3_amd64.deb ...
Unpacking libc-ares2:amd64 (1.18.1-1ubuntu0.22.04.3) ...
Selecting previously unselected package libaria2-0:amd64.
Preparing to unpack .../libaria2-0_1.36.0-1_amd64.deb ...
Unpacking libaria2-0:amd64 (1.36.0-1) ...
Selecting previously unselected package aria2.
Preparing to unpack .../aria2_1.36.0-1_amd64.deb ...
Unpacking aria2 (1.36.0-1) ...
Setting up libc-ares2:amd64 (1.18.1-1ubuntu0.22.04.3) ...
Setting up libaria2-0:amd64 (1.36.0-1) ...
Setting up aria2 (1.36.0-1) ...
Processing triggers for man-db (2.10.2-1) ...
Processing triggers for libc-bin (2.35-0ubuntu3.8) ...
/sbin/ldconfig.real: /usr/local/lib/libtcm_debug.so.1 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libhwloc.so.15 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libtbbmalloc.so.2 is not a 

[METADATA]The Physician (2013) [1080p] (metadata): 0.00B [00:00, ?B/s]

# OLD code #stable

In [ ]:
# Combined Code for Multi-Torrent Download with Progress Monitoring

# --- Step 1: Install necessary libraries and aria2c ---
!apt-get -qq install -y aria2
!pip install aria2p tqdm

print("✅ Installed required libraries and aria2c.")

# --- Step 2: Mount Google Drive ---
from google.colab import drive
drive.mount('/content/drive')

# --- Step 3: Set download directory in Google Drive ---
import os

# Change this path if you want another folder in Drive
download_dir = "/content/drive/My Drive/torrent_download"
os.makedirs(download_dir, exist_ok=True)

print("✅ Download directory set to:", download_dir)

# --- Step 4: Start aria2c with RPC enabled ---
import subprocess
import time

# Start aria2c daemon with RPC enabled
aria2_rpc_port = 6800  # You can change this port if needed
aria2_command = [
    "aria2c",
    "--enable-rpc",
    f"--rpc-listen-port={aria2_rpc_port}",
    "--rpc-listen-all",
    "--seed-time=0",
    "--console-log-level=warn",
    "-d", download_dir,
    "--continue=true", # Continue downloading partially downloaded files
    "--allow-overwrite=true", # Allow overwriting existing files
]

# Use subprocess.Popen to run aria2c in the background
# Redirect stdout and stderr to avoid cluttering the output directly
aria2_process = subprocess.Popen(
    aria2_command,
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
    text=True
)

# Give aria2c a moment to start
time.sleep(2)

# Check if aria2c process started successfully
if aria2_process.poll() is None:
    print(f"✅ aria2c daemon started successfully on port {aria2_rpc_port}")
else:
    print("❌ Failed to start aria2c daemon.")
    # Print any error output
    stderr_output = aria2_process.stderr.read()
    if stderr_output:
        print("aria2c stderr:\n", stderr_output)
    # Exit if aria2c failed to start
    exit()


# --- Step 5: Collect multiple magnet links ---
magnet_links = []
print("\nPlease enter the magnet links you want to download, one by one.")
print("Enter 'done' when you have finished adding links.")

while True:
    link = input(f"🔗 Enter magnet link {len(magnet_links) + 1} (or 'done'): ").strip()
    if link.lower() == 'done':
        break
    if link and link.startswith("magnet:?"):
        magnet_links.append(link)
        print("✅ Magnet link added.")
    elif link:
        print("❌ Invalid magnet link. Please make sure it starts with 'magnet:?'.")

if not magnet_links:
    print("\nNo magnet links provided. Exiting.")
    # Terminate aria2c process if no links were added
    if aria2_process.poll() is None:
        aria2_process.terminate()
    exit()

print(f"\nCollected {len(magnet_links)} magnet link(s).")


# --- Step 6: Add torrents via RPC ---
import aria2p

# Initialize aria2p client
# The default host is localhost and the default port is 6800
aria2 = aria2p.API(
    aria2p.Client(
        host="http://localhost",
        port=aria2_rpc_port,
        secret="" # No secret token is set in the aria2_command
    )
)

added_count = 0
error_count = 0

print("\nAdding torrents to aria2c...")

for link in magnet_links:
    try:
        result = aria2.add_magnet(link)
        if result:
            print(f"✅ Successfully added torrent: {result.name or result.gid}")
            added_count += 1
        else:
            print(f"❌ Failed to add torrent: {link}")
            error_count += 1
    except Exception as e:
        print(f"❌ Error adding torrent {link}: {e}")
        error_count += 1

print(f"\nSummary: {added_count} torrent(s) added successfully, {error_count} error(s).")

if added_count == 0:
    print("No torrents were added to monitor. Exiting.")
    # Terminate aria2c process if no links were successfully added
    if aria2_process.poll() is None:
        aria2_process.terminate()
    exit()

print("🚀 Starting download monitoring...")

# --- Step 7: Monitor download progress and display GUI-like progress ---
from tqdm.notebook import tqdm
import datetime

def get_downloads_status(api):
    """Fetches the status of all active, waiting, and paused downloads."""
    try:
        return api.get_downloads()
    except Exception as e:
        # Handle potential connection errors to aria2c
        print(f"Error fetching download status: {e}")
        return []


def display_progress(downloads):
    """Displays progress bars for each download."""
    if not downloads:
        # No downloads to display, but aria2c might still be running with completed/errored ones
        return

    # Create a dictionary to store tqdm bars, keyed by GID
    if not hasattr(display_progress, 'progress_bars'):
        display_progress.progress_bars = {}
        display_progress.completed_gids = set() # Keep track of completed downloads

    for download in downloads:
        gid = download.gid
        status = download.status
        total_length = download.total_length
        completed_length = download.completed_length
        download_speed = download.download_speed
        eta = download.eta

        # Convert bytes to MiB for display
        total_length_mib = total_length / (1024 * 1024) if total_length else 0
        completed_length_mib = completed_length / (1024 * 1024) if completed_length else 0
        download_speed_mibps = download_speed / (1024 * 1024) if download_speed else 0

        # Determine the state of the download
        is_complete = status == 'complete'
        is_error = status == 'error'
        is_removed = status == 'removed'

        if is_complete and gid not in display_progress.completed_gids:
            # Mark as completed and close the progress bar
            if gid in display_progress.progress_bars:
                display_progress.progress_bars[gid].n = total_length # Set to total for final display
                display_progress.progress_bars[gid].refresh()
                display_progress.progress_bars[gid].close()
                del display_progress.progress_bars[gid]
            print(f"\n✅ Download complete: {download.name or download.gid}")
            display_progress.completed_gids.add(gid)
            continue # Move to the next download

        if is_error or is_removed:
             if gid in display_progress.progress_bars:
                display_progress.progress_bars[gid].n = completed_length # Show current progress before closing
                display_progress.progress_bars[gid].refresh()
                display_progress.progress_bars[gid].close()
                del display_progress.progress_bars[gid]
             if is_error:
                 print(f"\n❌ Download error for {download.name or download.gid}. Status: {status}, Error Code: {download.error_code}, Error Message: {download.error_message}")
             elif is_removed:
                  print(f"\nℹ️ Download removed: {download.name or download.gid}")
             if gid in display_progress.completed_gids:
                 display_progress.completed_gids.remove(gid) # Remove from completed if it somehow ended up there
             continue


        if gid not in display_progress.progress_bars:
            # Create a new progress bar for a new download
            bar_description = f"{download.name or download.gid}"
            if total_length > 0:
                 display_progress.progress_bars[gid] = tqdm(total=total_length, unit='B', unit_scale=True, desc=bar_description, position=len(display_progress.progress_bars), leave=True)
            else:
                 # Handle case where total_length is not yet known (e.g., metadata download)
                 display_progress.progress_bars[gid] = tqdm(unit='B', unit_scale=True, desc=bar_description + " (metadata)", position=len(display_progress.progress_bars), leave=True)


        # Update the existing progress bar
        bar = display_progress.progress_bars[gid]
        bar.n = completed_length

        # Convert eta (timedelta) to seconds if it exists
        eta_seconds = eta.total_seconds() if isinstance(eta, datetime.timedelta) else None

        bar.set_postfix_str(f"Speed: {download_speed_mibps:.2f} MiB/s, ETA: {time.strftime('%H:%M:%S', time.gmtime(int(eta_seconds))) if eta_seconds is not None else 'N/A'}")
        bar.refresh()

# --- Monitoring loop ---
print("Monitoring downloads... Press the stop button to interrupt.")

try:
    while True:
        downloads = get_downloads_status(aria2)
        # Filter for downloads that are still in a state we want to monitor progress for
        active_monitor_downloads = [d for d in downloads if d.status in ['active', 'waiting', 'paused', 'downloading']]


        if not active_monitor_downloads and all(d.status in ['complete', 'error', 'removed'] for d in downloads):
             # Check if all downloads that were initially added are now in a final state
             # This handles cases where downloads might finish quickly before the first monitoring loop iteration
             print("\nAll downloads have finished or are in a final state.")
             break # Exit the loop if all downloads are finished

        display_progress(downloads) # Pass all downloads to handle state changes

        time.sleep(1)  # Update every second

except KeyboardInterrupt:
    print("\nMonitoring interrupted by user.")
except Exception as e:
    print(f"\nAn error occurred during monitoring: {e}")
finally:
    # Ensure all progress bars are closed when the loop exits
    if hasattr(display_progress, 'progress_bars'):
        for gid, bar in list(display_progress.progress_bars.items()): # Use list() to iterate over a copy
            bar.close()
            del display_progress.progress_bars[gid] # Clean up the dictionary

    # Terminate the aria2c process when monitoring stops
    if aria2_process.poll() is None:
        print("\nTerminating aria2c process...")
        aria2_process.terminate()
        aria2_process.wait()
        print("aria2c process terminated.")

    print("Monitoring stopped.")

Selecting previously unselected package libc-ares2:amd64.
(Reading database ... 126435 files and directories currently installed.)
Preparing to unpack .../libc-ares2_1.18.1-1ubuntu0.22.04.3_amd64.deb ...
Unpacking libc-ares2:amd64 (1.18.1-1ubuntu0.22.04.3) ...
Selecting previously unselected package libaria2-0:amd64.
Preparing to unpack .../libaria2-0_1.36.0-1_amd64.deb ...
Unpacking libaria2-0:amd64 (1.36.0-1) ...
Selecting previously unselected package aria2.
Preparing to unpack .../aria2_1.36.0-1_amd64.deb ...
Unpacking aria2 (1.36.0-1) ...
Setting up libc-ares2:amd64 (1.18.1-1ubuntu0.22.04.3) ...
Setting up libaria2-0:amd64 (1.36.0-1) ...
Setting up aria2 (1.36.0-1) ...
Processing triggers for man-db (2.10.2-1) ...
Processing triggers for libc-bin (2.35-0ubuntu3.8) ...
/sbin/ldconfig.real: /usr/local/lib/libtbbmalloc.so.2 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libtcm.so.1 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libtbbmalloc_proxy.so.2 is not

[METADATA]Americana.2023.1080p.10bit.WEBRip.6CH.x265.HEVC-PSA (metadata): 0.00B [00:00, ?B/s]

Americana.2023.1080p.10bit.WEBRip.6CH.x265.HEVC-PSA:   0%|          | 0.00/1.70G [00:00<?, ?B/s]


✅ Download complete: [METADATA]Americana.2023.1080p.10bit.WEBRip.6CH.x265.HEVC-PSA


[METADATA]Americana.2023.1080p.10bit.WEBRip.6CH.x265.HEVC-PSA:   0%|          | 0.00/8.34k [00:00<?, ?B/s]


All downloads have finished or are in a final state.

Terminating aria2c process...
aria2c process terminated.
Monitoring stopped.


# TESTING GROUNDS